# RAG Assignment: Baseline Retrieval vs. Cross-Encoder Re-Ranking

**Document:** EU General Data Protection Regulation (GDPR), Regulation (EU) 2016/679, the official PDF from EUR-Lex (88 pages, 173 recitals + 99 articles).
It is a long legal text full of general rules followed by exceptions ("shall not apply if..."), prohibitions and numeric limits, so it works well for stress testing retrieval.

**Two pipelines over the same chunks:**

| | Pipeline 1: Baseline | Pipeline 2: Re-Ranking |
|---|---|---|
| Step 1 | Bi-encoder (`all-MiniLM-L6-v2`) + FAISS, retrieve **Top 10** | Same Top 10 from Step 1 |
| Step 2 | Take the **Top 3** by cosine similarity | Score all 10 (query, chunk) pairs with the cross-encoder (`ms-marco-MiniLM-L-6-v2`), keep the **Top 3** |
| Step 3 | LLM (`Qwen2.5-7B-Instruct`) produces **Answer 1** | Same LLM, same prompt, produces **Answer 2** |

**Notebook flow:** load PDF → chunk → embed + index → load cross-encoder and LLM → define both pipelines → run the failure queries → rank-shift tables → compare answers → latency profiling → scaling experiment → save results.

**Hardware used:** 4 × RTX 2080 Ti (11 GB each). The 7B LLM is loaded in float16 (about 15 GB) and split across the GPUs with `device_map="auto"`.

## 0. Install dependencies
Run once (or use `pip install -r requirements.txt`).

In [ ]:
# !pip install -q pypdf sentence-transformers transformers accelerate faiss-cpu pandas

## 1. Imports and configuration
All the settings are in one place so they are easy to change.

In [17]:
import os, re, time
import numpy as np
import pandas as pd
import torch
import faiss
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM

PDF_PATH     = "data/gdpr.pdf"
EMBED_MODEL  = "sentence-transformers/all-MiniLM-L6-v2"   # bi-encoder
RERANK_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"     # cross-encoder
LLM_MODEL    = "Qwen/Qwen2.5-7B-Instruct"                 # open-weights LLM (Hugging Face)

CHUNK_SIZE    = 150   # words per chunk
CHUNK_OVERLAP = 30    # words shared by consecutive chunks (so sentences at a border are not lost)
TOP_K = 10            # candidates from the bi-encoder
TOP_N = 3             # chunks given to the LLM

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs("results", exist_ok=True)
pd.set_option("display.max_colwidth", 120)

print("Device:", DEVICE, "| GPUs:", torch.cuda.device_count())

Device: cuda | GPUs: 4


## 2. Load and clean the PDF
`pypdf` extracts the text page by page. Cleaning steps:
1. Drop the running page header ("... Official Journal of the European Union L 119/..."), which repeats on every page and is noise.
2. Join words that were hyphenated at a line break (`pro-\ncessing` becomes `processing`).
3. Collapse all whitespace into single spaces.

In [18]:
def clean_page(text):
    lines = [line for line in text.split("\n")
             if not ("Official Journal of the European Union" in line and "L 119/" in line)]
    text = "\n".join(lines)
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)   # re-join hyphenated words
    text = re.sub(r"\s+", " ", text)                # normalise whitespace
    return text.strip()

reader = PdfReader(PDF_PATH)
full_text = " ".join(clean_page(page.extract_text()) for page in reader.pages)

print("Pages:", len(reader.pages))
print("Words:", len(full_text.split()))
print(full_text[:500])

Pages: 88
Words: 54932
I (Legislative acts) REGULATIONS REGULATION (EU) 2016/679 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL of 27 April 2016 on the protection of natural persons with regard to the processing of personal data and on the free movement of such data, and repealing Directive 95/46/EC (General Data Protection Regulation) (Text with EEA relevance) THE EUROPEAN PARLIAMENT AND THE COUNCIL OF THE EUROPEAN UNION, Having regard to the Treaty on the Functioning of the European Union, and in particular Article 1


## 3. Chunking
I use fixed-size word windows (150 words, 30-word overlap). This is simple and gives every chunk a similar length.
150 words is roughly 200 tokens, which fits within the bi-encoder's 256-token limit, so the embedding does not silently cut off the end of a chunk.
**Both pipelines use exactly this chunk list.**

In [19]:
def chunk_text(text, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    words = text.split()
    step = size - overlap
    chunks = []
    for start in range(0, len(words), step):
        chunks.append(" ".join(words[start:start + size]))
        if start + size >= len(words):
            break
    return chunks

chunks = chunk_text(full_text)
print("Number of chunks:", len(chunks))
print("\nExample chunk 100:\n", chunks[100])

Number of chunks: 458

Example chunk 100:
 Regulation, including for the security of processing. The adherence of the processor to an approved code of conduct or an approved certification mechanism may be used as an element to demonstrate compliance with the obligations of the controller. The carrying-out of processing by a processor should be governed by a contract or other legal act under Union or Member State law, binding the processor to the controller, setting out the subject- matter and duration of the processing, the nature and purposes of the processing, the type of personal data and categories of data subjects, taking into account the specific tasks and responsibilities of the processor in the context of the processing to be carried out and the risk to the rights and freedoms of the data subject. The controller and processor may choose to use an individual contract or standard contractual clauses which are adopted either directly by the Commission


## 4. Pipeline 1, indexing: bi-encoder embeddings + FAISS vector store
* The bi-encoder embeds every chunk **independently** into a 384-dim vector. This is done **once, offline**.
* The vectors are L2-normalised, so the inner product equals **cosine similarity**. `IndexFlatIP` does an exact inner-product search.

In [20]:
embedder = SentenceTransformer(EMBED_MODEL, device=DEVICE)

# check that chunks fit inside the bi-encoder's max input length
n_tokens = [len(embedder.tokenizer(c)["input_ids"]) for c in chunks]
print(f"Max seq length: {embedder.max_seq_length} | longest chunk: {max(n_tokens)} tokens | "
      f"chunks over limit: {sum(n > embedder.max_seq_length for n in n_tokens)}")

chunk_embeddings = embedder.encode(chunks, batch_size=64, normalize_embeddings=True,
                                   show_progress_bar=True).astype("float32")

index = faiss.IndexFlatIP(chunk_embeddings.shape[1])
index.add(chunk_embeddings)
print("Vectors in index:", index.ntotal, "| dim:", chunk_embeddings.shape[1])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3388.47it/s]


Max seq length: 256 | longest chunk: 220 tokens | chunks over limit: 0


Batches: 100%|██████████| 8/8 [00:00<00:00, 17.19it/s]

Vectors in index: 458 | dim: 384


## 5. Load the cross-encoder (Pipeline 2)
The cross-encoder takes the query and a chunk **together** as one input (`[CLS] query [SEP] chunk [SEP]`) and outputs one relevance score.
It is more accurate but needs one forward pass per (query, chunk) pair, so we only run it on the 10 candidates.

In [21]:
reranker = CrossEncoder(RERANK_MODEL, device=DEVICE)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 3833.42it/s]


## 6. Load the LLM and define the prompt
* `Qwen2.5-7B-Instruct` in **float16** (Turing GPUs such as the RTX 2080 Ti and T4 have no native bfloat16). `device_map="auto"` spreads the layers over the available GPUs.
* Greedy decoding (`do_sample=False`) makes answers deterministic, so any difference between Answer 1 and Answer 2 comes **only from the retrieved chunks**.
* The prompt tells the model to use only the context. This matters because the LLM has probably seen the GDPR during training. Without the instruction it could answer from memory and hide a retrieval failure.

In [22]:
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
llm = AutoModelForCausalLM.from_pretrained(LLM_MODEL, torch_dtype=torch.float16, device_map="auto")
llm.eval()

PROMPT_TEMPLATE = """You are an assistant that answers questions about the EU General Data Protection Regulation (GDPR).
Answer the question using ONLY the context below. Do not use outside knowledge.
If the context does not contain the answer, reply exactly: "The provided context does not contain the answer."
Keep the answer short (2-4 sentences) and mention which chunk(s) you used.

Context:
{context}

Question: {question}
Answer:"""

def generate_answer(question, chunk_ids, max_new_tokens=250):
    context = "\n\n".join(f"[Chunk {i}] {chunks[i]}" for i in chunk_ids)
    messages = [{"role": "user", "content": PROMPT_TEMPLATE.format(context=context, question=question)}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(llm.device)
    with torch.no_grad():
        output = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    new_tokens = output[0][inputs["input_ids"].shape[1]:]          # drop the prompt tokens
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

Loading weights: 100%|██████████| 339/339 [00:03<00:00, 88.32it/s]


## 7. The two retrieval pipelines
* `baseline_retrieve`: embed the query, then FAISS search returns the Top 10 `(chunk_id, cosine score)`.
* `rerank`: score the same 10 `(query, chunk)` pairs with the cross-encoder and sort them by that score, highest first.

Pipeline 1 = `baseline_retrieve`, then Top 3, then `generate_answer`.
Pipeline 2 = `baseline_retrieve`, then `rerank`, then Top 3, then `generate_answer` (the same prompt).

In [23]:
def baseline_retrieve(query, k=TOP_K):
    q_emb = embedder.encode([query], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(q_emb, k)
    return [(int(i), float(s)) for i, s in zip(ids[0], scores[0])]

def rerank(query, candidates):
    pairs = [(query, chunks[cid]) for cid, _ in candidates]
    ce_scores = reranker.predict(pairs)
    order = np.argsort(-ce_scores)                      # descending by cross-encoder score
    return [(candidates[j][0], float(ce_scores[j])) for j in order]

## 8. Failure-mode test queries
Each query targets one failure type from the assignment. To measure ranks objectively, every query has one or more **gold phrases**: short verbatim pieces of the GDPR text that contain the true answer.
Every chunk containing one of these phrases counts as a "gold chunk". Some queries have several phrases for two reasons. The GDPR often states the same rule in an Article and again in its Recital (e.g. Art. 6(1) and Recital 47). And a long article can be split across two chunks (e.g. the list in Art. 9(2)). Matching ignores case and whitespace, so PDF spacing issues such as `20 000 000` vs `20000000` do not break it.

| Type | Idea |
|---|---|
| Keyword trap / distractor | Query words overlap an irrelevant passage strongly, while the answer uses different terms (e.g. "penalty" matches Art. 84 *Penalties*, but the maximum fine is in Art. 83 *administrative fines*) |
| Exceptions & corner cases | The general rule is easy to find. The "shall not apply..." exception sits in a different chunk |
| Negation & constraints | "not prohibited", "cannot be dismissed", "not below 13 years": bi-encoders tend to blur negative and positive statements |
| Other: buried chunk | Any case where the gold chunk is at rank 4-10 for the baseline and the cross-encoder moves it into the Top 3 (found automatically below) |

In [24]:
TEST_QUERIES = [
    # --- Keyword trap / distractor ---
    {"id": "Q1", "type": "Keyword trap",
     "query": "What is the maximum penalty a company can get for breaking the GDPR?",
     "gold": ["20 000 000 EUR"]},                                        # Art. 83(5)
    {"id": "Q2", "type": "Keyword trap",
     "query": "Our company was hacked and customer information leaked. How quickly must we tell the regulator?",
     "gold": ["not later than 72 hours"]},                               # Art. 33(1) / Recital 85
    {"id": "Q3", "type": "Keyword trap",
     "query": "Can a bank reject my loan application using only a computer algorithm, with no human involved?",
     "gold": ["based solely on automated processing",                    # Art. 22 / Recital 71
              "automatic refusal of an online credit application"]},
    # --- Exceptions & corner cases ---
    {"id": "Q4", "type": "Exception",
     "query": "Does a small business with 50 employees have to keep records of its processing activities?",
     "gold": ["fewer than 250"]},                                        # Art. 30(5) / Recital 13
    {"id": "Q5", "type": "Exception",
     "query": "If the stolen personal data was encrypted, does the company still have to inform the affected people about the breach?",
     "gold": ["unintelligible to any person who is not authorised"]},    # Art. 34(3)(a)
    {"id": "Q6", "type": "Exception",
     "query": "When can an organisation refuse to delete personal data even though the person asked for erasure?",
     "gold": ["for exercising the right of freedom of expression and information"]},  # Art. 17(3) / Recital 65
    {"id": "Q7", "type": "Exception",
     "query": "Can a public authority rely on legitimate interests as the legal basis when performing its tasks?",
     "gold": ["Point (f) of the first subparagraph shall not apply to processing carried out by public authorities",  # Art. 6(1)
              "that legal basis should not apply to the processing by public authorities"]},                     # Recital 47
    # --- Negation & constraints ---
    {"id": "Q8", "type": "Negation",
     "query": "In which cases is processing someone's health data NOT prohibited?",
     "gold": ["Paragraph 1 shall not apply if one of the following applies",          # Art. 9(2)
              "has given explicit consent to the processing of those personal data",    # Art. 9(2)(a)
              "Derogating from the prohibition on processing special categories"]},    # Recital 52
    {"id": "Q9", "type": "Negation",
     "query": "Is a company allowed to fire its data protection officer because it dislikes the officer's advice?",
     "gold": ["shall not be dismissed or penalised"]},                    # Art. 38(3)
    {"id": "Q10", "type": "Negation",
     "query": "Can a 12-year-old give consent alone to sign up for a social media app?",
     "gold": ["not below 13 years"]},                                    # Art. 8(1)
]

def normalise(s):
    return re.sub(r"\s+", "", s.lower())

def find_gold_chunks(phrases):
    # a chunk is gold if it contains any of the gold phrases
    return [i for i, c in enumerate(chunks) if any(normalise(p) in normalise(c) for p in phrases)]

for t in TEST_QUERIES:
    t["gold_ids"] = find_gold_chunks(t["gold"])
    print(f'{t["id"]:>4} gold chunks: {t["gold_ids"]}' + ("   <-- WARNING: phrase not found" if not t["gold_ids"] else ""))

  Q1 gold chunks: [436, 437, 438]
  Q2 gold chunks: [105, 293, 294]
  Q3 gold chunks: [84, 85, 267, 268, 346, 347]
  Q4 gold chunks: [12, 290]
  Q5 gold chunks: [297]
  Q6 gold chunks: [77, 259]
  Q7 gold chunks: [52, 223]
  Q8 gold chunks: [60, 61, 230, 231]
  Q9 gold chunks: [311]
 Q10 gold chunks: [229]


## 9. Run both pipelines on every test query
For each query we record:
* the Top 10 ranking from the bi-encoder and the new ranking after the cross-encoder (the **rank shift**),
* the rank of the gold chunk in each ranking,
* Answer 1 and Answer 2,
* the time taken by each stage.

Before the loop, one warm-up call runs each model once. The first GPU call is slow because CUDA kernels are loaded, and that would distort the timings.

In [25]:
# warm-up: so CUDA initialisation is not counted in the latency numbers
_ = generate_answer("warm up", [c for c, _ in rerank("warm up", baseline_retrieve("warm up"))[:TOP_N]], max_new_tokens=5)

def gold_rank(ranking, gold_ids):
    for rank, (cid, _) in enumerate(ranking, start=1):
        if cid in gold_ids:
            return rank
    return None   # gold chunk is not in the list at all

results, rank_tables = [], {}

for t in TEST_QUERIES:
    q = t["query"]
    t0 = time.perf_counter(); top10 = baseline_retrieve(q)
    t1 = time.perf_counter(); reranked = rerank(q, top10)
    t2 = time.perf_counter(); ans1 = generate_answer(q, [c for c, _ in top10[:TOP_N]])
    t3 = time.perf_counter(); ans2 = generate_answer(q, [c for c, _ in reranked[:TOP_N]])
    t4 = time.perf_counter()

    # rank-shift table for this query
    new_rank = {cid: r for r, (cid, _) in enumerate(reranked, start=1)}
    ce_score = dict(reranked)
    rows = []
    for old_r, (cid, cos) in enumerate(top10, start=1):
        rows.append({"chunk": cid, "baseline_rank": old_r, "cosine": round(cos, 3),
                     "rerank_rank": new_rank[cid], "ce_score": round(ce_score[cid], 3),
                     "shift": f"#{old_r} -> #{new_rank[cid]}",
                     "gold": "YES" if cid in t["gold_ids"] else "",
                     "text": chunks[cid][:110] + "..."})
    rank_tables[t["id"]] = pd.DataFrame(rows)

    results.append({"id": t["id"], "type": t["type"], "query": q, "gold_ids": t["gold_ids"],
                    "gold_rank_baseline": gold_rank(top10, t["gold_ids"]),
                    "gold_rank_reranked": gold_rank(reranked, t["gold_ids"]),
                    "top3_baseline": [c for c, _ in top10[:TOP_N]],
                    "top3_reranked": [c for c, _ in reranked[:TOP_N]],
                    "answer_1": ans1, "answer_2": ans2,
                    "t_retrieval_ms": (t1 - t0) * 1000, "t_rerank_ms": (t2 - t1) * 1000,
                    "t_gen1_s": t3 - t2, "t_gen2_s": t4 - t3})
    print("done", t["id"])

results_df = pd.DataFrame(results)
# integer ranks, with <NA> when the gold chunk is not in the top-10
results_df[["gold_rank_baseline", "gold_rank_reranked"]] = results_df[["gold_rank_baseline", "gold_rank_reranked"]].astype("Int64")

done Q1
done Q2
done Q3
done Q4
done Q5
done Q6
done Q7
done Q8
done Q9
done Q10


## 10. Rank-shift analysis
### 10.1 Summary: where did the gold chunk end up?
* `gold_rank_baseline`: best rank of a gold chunk in the bi-encoder Top 10 (`None` means it was not in the Top 10 at all).
* `gold_rank_reranked`: its rank after the cross-encoder.
* **Outcome**:
  * *Restored*: the gold chunk was outside the baseline Top 3 and the re-ranker moved it into the Top 3.
  * *Both OK*: it was already in the baseline Top 3 and stayed there.
  * *Degraded*: the re-ranker pushed it out of the Top 3.
  * *Not recoverable*: it was not in the Top 10, so the re-ranker never saw it.

In [26]:
def outcome(row):
    b, r = row["gold_rank_baseline"], row["gold_rank_reranked"]
    if pd.isna(b):                return "Not recoverable (gold not in top-10)"
    if b > TOP_N and r <= TOP_N: return "Restored by re-ranker"
    if b <= TOP_N and r <= TOP_N: return "Both OK"
    if b <= TOP_N and r > TOP_N:  return "Degraded by re-ranker"
    return "Both fail"

results_df["outcome"] = results_df.apply(outcome, axis=1)
summary = results_df[["id", "type", "query", "gold_ids", "gold_rank_baseline", "gold_rank_reranked", "outcome"]]
display(summary)

hit3_base = (results_df["gold_rank_baseline"] <= TOP_N).sum()   # <NA> rows are not counted
hit3_rr   = (results_df["gold_rank_reranked"] <= TOP_N).sum()
print(f"Gold chunk in Top-3  ->  Baseline: {hit3_base}/{len(results_df)}   Re-ranked: {hit3_rr}/{len(results_df)}")

,id,type,query,gold_ids,gold_rank_baseline,gold_rank_reranked,outcome
0,Q1,Keyword trap,What is the maximum penalty a company can get for breaking the GDPR?,"[436, 437, 438]",1,1,Both OK
1,Q2,Keyword trap,Our company was hacked and customer information leaked. How quickly must we tell the regulator?,"[105, 293, 294]",1,1,Both OK
2,Q3,Keyword trap,"Can a bank reject my loan application using only a computer algorithm, with no human involved?","[84, 85, 267, 268, 346, 347]",2,1,Both OK
3,Q4,Exception,Does a small business with 50 employees have to keep records of its processing activities?,"[12, 290]",2,1,Both OK
4,Q5,Exception,"If the stolen personal data was encrypted, does the company still have to inform the affected people about the breach?",[297],1,1,Both OK
5,Q6,Exception,When can an organisation refuse to delete personal data even though the person asked for erasure?,"[77, 259]",6,3,Restored by re-ranker
6,Q7,Exception,Can a public authority rely on legitimate interests as the legal basis when performing its tasks?,"[52, 223]",4,1,Restored by re-ranker
7,Q8,Negation,In which cases is processing someone's health data NOT prohibited?,"[60, 61, 230, 231]",5,1,Restored by re-ranker
8,Q9,Negation,Is a company allowed to fire its data protection officer because it dislikes the officer's advice?,[311],1,1,Both OK
9,Q10,Negation,Can a 12-year-old give consent alone to sign up for a social media app?,[229],1,1,Both OK


Gold chunk in Top-3  ->  Baseline: 7/10   Re-ranked: 10/10


### 10.2 Detailed rank-shift table per query
`shift` shows how each chunk moved, e.g. `#8 -> #1`. The gold chunk is marked `YES`.

In [27]:
for t in TEST_QUERIES:
    row = results_df.set_index("id").loc[t["id"]]
    print(f'\n=== {t["id"]} [{t["type"]}] {t["query"]}')
    print(f'    gold rank: baseline {row["gold_rank_baseline"]}  ->  re-ranked {row["gold_rank_reranked"]}   ({row["outcome"]})')
    display(rank_tables[t["id"]])


=== Q1 [Keyword trap] What is the maximum penalty a company can get for breaking the GDPR?
    gold rank: baseline 1  ->  re-ranked 1   (Both OK)


,chunk,baseline_rank,cosine,rerank_rank,ce_score,shift,gold,text
0,436,1,0.609,1,-7.263,#1 -> #1,YES,"infringes several provisions of this Regulation, the total amount of the administrative fine shall not exceed ..."
1,437,2,0.558,2,-7.447,#2 -> #2,YES,"following provisions shall, in accordance with paragraph 2, be subject to administrative fines up to 20 000 00..."
2,438,3,0.537,3,-8.350,#3 -> #3,YES,processing or the suspension of data flows by the supervisory authority pursuant to Article 58(2) or failure t...
3,176,4,0.489,5,-10.322,#4 -> #5,,"damage suffered, degree of responsibility or any relevant previous infringements, the manner in which the infr..."
4,432,5,0.479,6,-10.869,#5 -> #6,,controller or processor shall be held liable for the entire damage in order to ensure effective compensation o...
5,201,6,0.476,10,-11.325,#6 -> #10,,"as set out in that Article, this Regulation does not go beyond what is necessary in order to achieve that obje..."
6,177,7,0.469,4,-9.553,#7 -> #4,,"deprivation of the profits obtained through infringements of this Regulation. However, the imposition of crimi..."
7,435,8,0.460,7,-11.026,#8 -> #7,,"the manner in which the infringement became known to the supervisory authority, in particular whether, and if ..."
8,373,9,0.447,8,-11.240,#9 -> #8,,certification body pursuant to Article 43; (r) authorise contractual clauses and provisions referred to in Art...
9,414,10,0.439,9,-11.294,#10 -> #9,,and on further necessary requirements to ensure the protection of personal data of the data subjects concerned...



=== Q2 [Keyword trap] Our company was hacked and customer information leaked. How quickly must we tell the regulator?
    gold rank: baseline 1  ->  re-ranked 1   (Both OK)


,chunk,baseline_rank,cosine,rerank_rank,ce_score,shift,gold,text
0,294,1,0.470,3,-9.032,#1 -> #3,YES,"controller shall without undue delay and, where feasible, not later than 72 hours after having become aware of..."
1,105,2,0.468,1,-7.861,#2 -> #1,YES,"person concerned. Therefore, as soon as the controller becomes aware that a personal data breach has occurred,..."
2,305,3,0.440,4,-9.922,#3 -> #4,,"is of the opinion that the intended processing referred to in paragraph 1 would infringe this Regulation, in p..."
3,107,4,0.438,6,-10.231,#4 -> #6,,to implement appropriate measures against continuing or similar personal data breaches may justify more time f...
4,104,5,0.436,2,-9.012,#5 -> #2,,with this Regulation. Where a data-protection impact assessment indicates that processing operations involve a...
5,295,6,0.434,5,-10.168,#6 -> #5,,approximate number of data subjects concerned and the categories and approximate number of personal data recor...
6,412,7,0.417,8,-10.628,#7 -> #8,,"for erasing links, copies or replications of personal data from publicly available communication services as r..."
7,296,8,0.415,9,-10.646,#8 -> #9,,"relating to the personal data breach, its effects and the remedial action taken. That documentation shall enab..."
8,106,9,0.407,7,-10.592,#9 -> #7,,"breach, without undue delay, where that personal data breach is likely to result in a high risk to the rights ..."
9,398,10,0.403,10,-10.963,#10 -> #10,,"to in paragraphs 1 and 2, the Board shall issue an opinion on the matter submitted to it provided that it has ..."



=== Q3 [Keyword trap] Can a bank reject my loan application using only a computer algorithm, with no human involved?
    gold rank: baseline 2  ->  re-ranked 1   (Both OK)


...

,chunk,baseline_rank,cosine,rerank_rank,ce_score,shift,gold,text
0,73,1,0.709,4,0.847,#1 -> #4,,expressly laid down by law or where the provision of information to the data subject proves to be impossible o...
1,74,2,0.675,2,1.094,#2 -> #2,,"processing. This includes the right for data subjects to have access to data concerning their health, for exam..."
2,233,3,0.670,6,-1.548,#3 -> #6,,personal data which are manifestly made public by the data subject; (f) processing is necessary for the establ...
3,70,4,0.651,7,-1.920,#4 -> #7,,and of the Council of 16 December 2008 on Community statistics on public health and health and safety at work ...
4,231,5,0.648,5,-0.110,#5 -> #5,YES,one of the following applies: (a) the data subject has given explicit consent to the processing of those perso...
5,64,6,0.642,3,1.055,#6 -> #3,,"needs, in particular where the processing of such data is carried out for certain health-related purposes by p..."
6,61,7,0.633,1,2.610,#7 -> #1,YES,to permit the exercise of fundamental freedoms. (52) Derogating from the prohibition on processing special cat...
7,43,8,0.630,8,-2.191,#8 -> #8,,"of personal data and the equipment used for the processing. (40) In order for processing to be lawful, persona..."
8,346,9,0.618,10,-4.273,#9 -> #10,,"including the categories of personal data, the type of processing and its purposes, the type of data subjects ..."
9,9,10,0.613,9,-3.690,#10 -> #9,,natural persons with regard to the processing of personal data should be ensured throughout the Union. Regardi...



=== Q9 [Negation] Is a company allowed to fire its data protection officer because it dislikes the officer's advice?
    gold rank: baseline 1  ->  re-ranked 1   (Both OK)


,chunk,baseline_rank,cosine,rerank_rank,ce_score,shift,gold,text
0,311,1,0.587,1,-2.441,#1 -> #1,YES,data protection officer in performing the tasks referred to in Article 39 by providing resources necessary to ...
1,309,2,0.580,3,-2.984,#2 -> #3,,officer provided that a data protection officer is easily accessible from each establishment. 3. Where the con...
2,312,3,0.571,2,-2.717,#3 -> #2,,the exercise of their rights under this Regulation. 5. The data protection officer shall be bound by secrecy o...
3,313,4,0.567,4,-3.144,#4 -> #4,,"provisions; (b) to monitor compliance with this Regulation, with other Union or Member State data protection p..."
4,374,5,0.565,6,-3.455,#5 -> #6,,"data subject and, where applicable, for the data protection officer. 4. Where requests are manifestly unfounde..."
5,373,6,0.564,8,-5.504,#6 -> #8,,certification body pursuant to Article 43; (r) authorise contractual clauses and provisions referred to in Art...
6,310,7,0.562,5,-3.237,#7 -> #5,,"and, in particular, expert knowledge of data protection law and practices and the ability to fulfil the tasks ..."
7,104,8,0.534,7,-5.284,#8 -> #7,,with this Regulation. Where a data-protection impact assessment indicates that processing operations involve a...
8,293,9,0.521,10,-10.448,#9 -> #10,,approved code of conduct as referred to in Article 40 or an approved certification mechanism as referred to in...
9,107,10,0.518,9,-7.291,#10 -> #9,,to implement appropriate measures against continuing or similar personal data breaches may justify more time f...



=== Q10 [Negation] Can a 12-year-old give consent alone to sign up for a social media app?
    gold rank: baseline 1  ->  re-ranked 1   (Both OK)


,chunk,baseline_rank,cosine,rerank_rank,ce_score,shift,gold,text
0,229,1,0.410,1,-7.418,#1 -> #1,YES,"in relation to information society services 1. Where point (a) of Article 6(1) applies, in relation to the off..."
1,228,2,0.388,3,-9.868,#2 -> #3,,an infringement of this Regulation shall not be binding. 3. The data subject shall have the right to withdraw ...
2,68,3,0.384,9,-11.195,#3 -> #9,,"concise, easily accessible and easy to understand, and that clear and plain language and, additionally, where ..."
3,230,4,0.368,5,-10.710,#4 -> #5,,"consent is given or authorised by the holder of parental responsibility over the child, taking into considerat..."
4,40,5,0.363,7,-10.872,#5 -> #7,,to children when using services offered directly to a child. The consent of the holder of parental responsibil...
5,45,6,0.352,6,-10.788,#6 -> #6,,should ensure that the data subject is aware of the fact that and the extent to which consent is given. In acc...
6,39,7,0.349,8,-11.119,#7 -> #8,,"undertakings by virtue, for example, of ownership, financial participation or the rules which govern it or the..."
7,238,8,0.332,10,-11.197,#8 -> #10,,"except where the data subject, for the purpose of exercising his or her rights under those articles, provides ..."
8,32,9,0.327,2,-9.702,#9 -> #2,,"relating to him or her, such as by a written statement, including by electronic means, or an oral statement. T..."
9,77,10,0.321,4,-10.212,#10 -> #4,,"has withdrawn his or her consent or objects to the processing of personal data concerning him or her, or where..."


### 10.3 "Buried chunk" cases (other document-specific failures)
Queries where the gold chunk was buried at **rank 4-10** by the bi-encoder and promoted into the Top 3 by the cross-encoder.

In [28]:
buried = results_df[results_df["outcome"] == "Restored by re-ranker"]
for _, row in buried.iterrows():
    print(f'{row["id"]}: gold chunk moved from rank #{row["gold_rank_baseline"]} -> #{row["gold_rank_reranked"]}  | {row["query"]}')
print(f"\n{len(buried)} queries restored by the re-ranker.")

Q6: gold chunk moved from rank #6 -> #3  | When can an organisation refuse to delete personal data even though the person asked for erasure?
Q7: gold chunk moved from rank #4 -> #1  | Can a public authority rely on legitimate interests as the legal basis when performing its tasks?
Q8: gold chunk moved from rank #5 -> #1  | In which cases is processing someone's health data NOT prohibited?

3 queries restored by the re-ranker.


## 11. Answer 1 vs Answer 2
For each query: the gold chunk text (the ground truth), then both answers.
When reading them, check for: a **hallucination** (a made-up rule or number), an **omitted clause** (the exception or condition is missing), or a **misstated rule** (e.g. saying something is always prohibited).

In [29]:
for t in TEST_QUERIES:
    row = results_df.set_index("id").loc[t["id"]]
    print("=" * 110)
    print(f'{t["id"]} [{t["type"]}]  {t["query"]}')
    print(f'Gold chunk(s) {t["gold_ids"]}   baseline top-3: {row["top3_baseline"]}   re-ranked top-3: {row["top3_reranked"]}')
    if t["gold_ids"]:
        print("\nGOLD TEXT:", chunks[t["gold_ids"][0]][:600], "...")
    print("\nANSWER 1 (baseline):\n", row["answer_1"])
    print("\nANSWER 2 (re-ranked):\n", row["answer_2"])

Q1 [Keyword trap]  What is the maximum penalty a company can get for breaking the GDPR?
Gold chunk(s) [436, 437, 438]   baseline top-3: [436, 437, 438]   re-ranked top-3: [436, 437, 438]

GOLD TEXT: infringes several provisions of this Regulation, the total amount of the administrative fine shall not exceed the amount specified for the gravest infringement. 4. Infringements of the following provisions shall, in accordance with paragraph 2, be subject to administrative fines up to 10 000 000 EUR, or in the case of an undertaking, up to 2 % of the total worldwide annual turnover of the preceding financial year, whichever is higher: (a) the obligations of the controller and the processor pursuant to Articles 8, 11, 25 to 39 and 42 and 43; (b) the obligations of the certification body pursuan ...

ANSWER 1 (baseline):
 The maximum penalty a company can get for breaking the GDPR is up to 20,000,000 EUR, or in the case of an undertaking, up to 4% of the total worldwide annual turnover of the

## 12. Latency profiling
Retrieval and re-ranking take only milliseconds, so a single measurement is noisy. Each stage is repeated **20 times per query** and averaged over all 10 queries. The models are already warm at this point.
LLM generation time comes from the run in Section 9.

In [30]:
REPEATS = 20
ret_times, rr_times = [], []
for t in TEST_QUERIES:
    for _ in range(REPEATS):
        t0 = time.perf_counter(); top10 = baseline_retrieve(t["query"])
        t1 = time.perf_counter(); rerank(t["query"], top10)
        t2 = time.perf_counter()
        ret_times.append((t1 - t0) * 1000)
        rr_times.append((t2 - t1) * 1000)

ret_ms, rr_ms = np.mean(ret_times), np.mean(rr_times)
gen1_s, gen2_s = results_df["t_gen1_s"].mean(), results_df["t_gen2_s"].mean()

latency = pd.DataFrame({
    "Stage": ["Dense retrieval (embed query + FAISS top-10)", "Cross-encoder re-rank (10 pairs)",
              "Total retrieval", "LLM generation (avg)", "End-to-end (avg)"],
    "Pipeline 1: Baseline": [f"{ret_ms:.2f} ms", "-", f"{ret_ms:.2f} ms",
                             f"{gen1_s:.2f} s", f"{gen1_s + ret_ms/1000:.2f} s"],
    "Pipeline 2: Re-rank":  [f"{ret_ms:.2f} ms", f"{rr_ms:.2f} ms", f"{ret_ms + rr_ms:.2f} ms",
                             f"{gen2_s:.2f} s", f"{gen2_s + (ret_ms + rr_ms)/1000:.2f} s"],
})
display(latency)
print(f"Re-ranking adds {rr_ms:.2f} ms ({rr_ms / ret_ms:.1f}x the retrieval time), "
      f"which is {100 * rr_ms / 1000 / gen2_s:.2f}% of the LLM generation time.")

,Stage,Pipeline 1: Baseline,Pipeline 2: Re-rank
0,Dense retrieval (embed query + FAISS top-10),8.02 ms,8.02 ms
1,Cross-encoder re-rank (10 pairs),-,16.17 ms
2,Total retrieval,8.02 ms,24.19 ms
3,LLM generation (avg),4.38 s,3.86 s
4,End-to-end (avg),4.38 s,3.89 s


Re-ranking adds 16.17 ms (2.0x the retrieval time), which is 0.42% of the LLM generation time.


## 13. Scaling experiment: why not run the cross-encoder on every chunk?
We compare, for one query:
* **Bi-encoder:** the chunk embeddings are precomputed, so at query time we only embed the query and do one matrix search.
* **Cross-encoder on all chunks:** it needs a full transformer forward pass for **every** (query, chunk) pair, on every query, and none of this can be precomputed.

The result is then extrapolated linearly to bigger collections.

In [31]:
q = TEST_QUERIES[0]["query"]

t0 = time.perf_counter(); baseline_retrieve(q); t_bi = time.perf_counter() - t0
t0 = time.perf_counter(); reranker.predict([(q, c) for c in chunks], batch_size=64); t_ce_all = time.perf_counter() - t0
t0 = time.perf_counter(); rerank(q, baseline_retrieve(q)); t_two_stage = time.perf_counter() - t0

print(f"Corpus size: {len(chunks)} chunks")
print(f"Bi-encoder retrieval            : {t_bi*1000:8.2f} ms")
print(f"Two-stage (bi-encoder + CE@10)  : {t_two_stage*1000:8.2f} ms")
print(f"Cross-encoder over ALL chunks   : {t_ce_all*1000:8.2f} ms")

per_pair = t_ce_all / len(chunks)
for n in [10_000, 1_000_000]:
    print(f"Estimated cross-encoder time for {n:>9,} chunks: {per_pair * n:10.1f} s per query")

Corpus size: 458 chunks
Bi-encoder retrieval            :     9.60 ms
Two-stage (bi-encoder + CE@10)  :    22.96 ms
Cross-encoder over ALL chunks   :   348.59 ms
Estimated cross-encoder time for    10,000 chunks:        7.6 s per query
Estimated cross-encoder time for 1,000,000 chunks:      761.1 s per query


## 14. Save results
These files are used in the report.

In [33]:
results_df.to_csv("results/summary.csv", index=False)
latency.to_csv("results/latency.csv", index=False)
pd.concat([df.assign(query_id=qid) for qid, df in rank_tables.items()]).to_csv("results/rank_shifts.csv", index=False)

with open("results/answers.md", "w") as f:
    for _, row in results_df.iterrows():
        f.write(f'## {row["id"]} ({row["type"]}): {row["query"]}\n\n')
        f.write(f'- Gold chunks: {row["gold_ids"]}\n')
        f.write(f'- Gold rank: baseline {row["gold_rank_baseline"]} -> re-ranked {row["gold_rank_reranked"]} ({row["outcome"]})\n')
        f.write(f'- Top-3 baseline: {row["top3_baseline"]} | Top-3 re-ranked: {row["top3_reranked"]}\n\n')
        f.write(f'**Answer 1 (baseline):** {row["answer_1"]}\n\n')
        f.write(f'**Answer 2 (re-ranked):** {row["answer_2"]}\n\n---\n\n')

print("Saved: results/summary.csv, results/latency.csv, results/rank_shifts.csv, results/answers.md")

Saved: results/summary.csv, results/latency.csv, results/rank_shifts.csv, results/answers.md


## 15. Conclusion
The analysis (architecture comparison, rank shifts, answer comparison, latency and the scaling question) is in the separate report.